In [1]:
import os
import random
import shutil
from ultralytics import YOLO

In [3]:
dataset_root = r"../Data/cctv-weapon-dataset/Dataset"
images_dir = os.path.join(dataset_root, "images")
labels_dir = os.path.join(dataset_root, "labels")

print("Images:", len(os.listdir(images_dir)))
print("Labels:", len(os.listdir(labels_dir)))

Images: 141
Labels: 141


In [4]:
#Split dataset into train/val/test (80/10/10)
def split_dataset(images_dir, labels_dir, output_dir, train_ratio=0.8, val_ratio=0.1):
    os.makedirs(output_dir, exist_ok=True)
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(output_dir, "images", split), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "labels", split), exist_ok=True)

    images = [f for f in os.listdir(images_dir) if f.endswith((".jpg", ".png"))]
    random.shuffle(images)

    train_cutoff = int(len(images) * train_ratio)
    val_cutoff = int(len(images) * (train_ratio + val_ratio))

    splits = {
        "train": images[:train_cutoff],
        "val": images[train_cutoff:val_cutoff],
        "test": images[val_cutoff:]
    }

    for split, files in splits.items():
        for img in files:
            label = os.path.splitext(img)[0] + ".txt"
            shutil.copy(os.path.join(images_dir, img), os.path.join(output_dir, "images", split, img))
            shutil.copy(os.path.join(labels_dir, label), os.path.join(output_dir, "labels", split, label))

    print("Split completed:", {k: len(v) for k, v in splits.items()})

split_dataset(images_dir, labels_dir, os.path.join(dataset_root, "yolo-split"))

Split completed: {'train': 112, 'val': 14, 'test': 15}


In [8]:
yaml_content = """
path: ../Data/cctv-weapon-dataset/Dataset/yolo-split
train: images/train
val: images/val
test: images/test

names:
  0: person
  1: weapon
"""

yaml_path = os.path.join(dataset_root, "yolo-split", "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("dataset.yaml created at:", yaml_path)

dataset.yaml created at: D:/Studies/Self/CCTV-Threat-Detection/Data/cctv-weapon-dataset/Dataset\yolo-split\dataset.yaml


In [9]:
model = YOLO("yolov8n.pt")

def save_best_weights(trainer):
    save_path = os.path.join("Notebooks", "weapon-detection-model-simuletic.pt")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    shutil.copy(trainer.best, save_path)

model.add_callback("on_train_end", save_best_weights)

model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16
)

Ultralytics 8.4.14  Python-3.10.6 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3080, 10240MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/Studies/Self/CCTV-Threat-Detection/Data/cctv-weapon-dataset/Dataset\yolo-split\dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001BEAB4AB700>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

In [10]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.14  Python-3.10.6 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3080, 10240MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2365.4269.8 MB/s, size: 1543.8 KB)
val: Scanning D:\Studies\Self\CCTV-Threat-Detection\Data\cctv-weapon-dataset\Dataset\yolo-split\labels\val.cache... 39 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 39/39  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.7s/it 5.1s2.0s3s
                   all         39         75      0.958      0.964      0.984      0.801
                person         38         38      0.949      0.982      0.993      0.869
                weapon         37         37      0.966      0.946      0.974      0.734
Speed: 9.0ms preprocess, 30.7ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to D:\Studies\Self\CCTV-Threat-Detection\Notebooks\runs\detect\v

In [11]:
print(model.names)

{0: 'person', 1: 'weapon'}


In [12]:
results = model.predict(source=os.path.join(dataset_root, "yolo-split/images/test"), save=True)


image 1/40 D:\Studies\Self\CCTV-Threat-Detection\Data\cctv-weapon-dataset\Dataset\yolo-split\images\test\Scene1_2.png: 640x640 1 person, 1 weapon, 17.7ms
image 2/40 D:\Studies\Self\CCTV-Threat-Detection\Data\cctv-weapon-dataset\Dataset\yolo-split\images\test\Scene1_5.png: 640x640 1 person, 1 weapon, 20.1ms
image 3/40 D:\Studies\Self\CCTV-Threat-Detection\Data\cctv-weapon-dataset\Dataset\yolo-split\images\test\Scene1_9.png: 640x640 1 person, 1 weapon, 22.3ms
image 4/40 D:\Studies\Self\CCTV-Threat-Detection\Data\cctv-weapon-dataset\Dataset\yolo-split\images\test\Scene2_10.png: 640x640 1 person, 2 weapons, 18.7ms
image 5/40 D:\Studies\Self\CCTV-Threat-Detection\Data\cctv-weapon-dataset\Dataset\yolo-split\images\test\Scene2_13.png: 640x640 1 person, 1 weapon, 18.0ms
image 6/40 D:\Studies\Self\CCTV-Threat-Detection\Data\cctv-weapon-dataset\Dataset\yolo-split\images\test\Scene2_15.png: 640x640 1 person, 2 weapons, 22.6ms
image 7/40 D:\Studies\Self\CCTV-Threat-Detection\Data\cctv-weapon-data